# Hire cost and marginal ROI

Cost to hire the `n`-th additional hand on a given day is `farmHandCostMult * fib(n)`, where the Fibonacci sequence starts 1, 1, 2, 3, 5, 8, 13, 21, .... With the default multiplier of 1 the sequence is exactly that. The counter resets at the start of each day.

Each hand gets 24 actions per day. A hand recovers its own cost if its marginal production more than offsets that day's hire cost.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kaggriculture.env.constants import MARKET_PARAMS

FIG_DIR = Path.cwd().parent / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def fib(n: int) -> int:
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a


def hire_costs(n_hires: int, mult: int = 1) -> list[int]:
    return [mult * fib(i) for i in range(n_hires)]


costs = hire_costs(10)
cumulative = np.cumsum(costs)
pd.DataFrame({"nth_hire": range(1, 11), "cost": costs, "cumulative_cost": cumulative})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(1, 11)
ax.bar(x, costs, color="#1f77b4", label="cost of nth hand")
ax.plot(x, cumulative, color="#d62728", marker="o", label="cumulative daily hire cost")
ax.set_xticks(x)
ax.set_xlabel("nth hand hired that day")
ax.set_ylabel("cost ($)")
ax.set_title("Hire cost (Fibonacci) per additional hand")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "hire-cost.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# Break-even: how many units of $X product must each hand produce (net) to pay for itself
# on the day it is hired?
resources = {
    "WHEAT": MARKET_PARAMS["WHEAT"]["base"],
    "CARROT": MARKET_PARAMS["CARROT"]["base"],
    "MELON": MARKET_PARAMS["MELON"]["base"],
    "STRAWBERRY": MARKET_PARAMS["STRAWBERRY"]["base"],
    "EGG": MARKET_PARAMS["EGG"]["base"],
}
rows = []
for n in range(1, 11):
    cost = fib(n - 1)
    row = {"nth_hand": n, "cost": cost}
    for resource, price in resources.items():
        row[f"units_of_{resource}_to_breakeven"] = round(cost / price, 2)
    rows.append(row)

pd.DataFrame(rows).set_index("nth_hand")

## Takeaways

- First hand costs $1. Even one wheat harvest pays for it 25x over. Almost always worth it once you have any productive activity to do.
- 2nd hand: also $1. Same conclusion.
- 3rd hand: $2. Still trivial.
- 4th hand: $3.
- 5th hand: $5.
- 6th hand: $8. Marginal decision zone begins here.
- 7th hand: $13. Break-even is 1 carrot or half a melon; if the hand cannot even do one harvest, skip.
- 8th hand: $21.
- 9th hand: $34.
- 10th hand: $55.

Cumulative daily hire cost through the 10th hand is $143. Rarely worth it unless the plan needs to squeeze extra actions on a market-timing day (many sells to line up before a price drop, for example).

In practice the meta at the top of the leaderboard uses 2 to 5 hands most days, ramping up mid-season when the tile count has grown past what the main farmer + a couple of hands can service.